# 09. Relational Joins, Merges & Concatenation: Beginner Guide

### 🌟 What Are Relational Merges, Joins & Concatenations in Pandas?
Data is often split across multiple tables. Pandas provides relational database join operations (`pd.merge()` supporting inner, left, right, and outer joins) and axis stacking (`pd.concat()`) to combine datasets seamlessly without manual loops.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Relational Merges (`pd.merge`)**: Covers inner, left, right, and outer joins.
- **Vertical Stacking (`axis=0`)**: Covers `pd.concat([df1, df2], axis=0)`.
- **Horizontal Stacking (`axis=1`)**: Covers `pd.concat([df1, df2], axis=1)`.
- **Index-Based Joins (`df.join`)**: Covers `df1.join(df2)`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Relational Database Merges with `pd.merge()`
Enriches transactions with mock customer profile metadata. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Always check `df.shape` before and after merging to verify you didn't accidentally introduce Cartesian duplication or drop rows.

**Syntax:** `pd.merge(df, customer_metadata, on='customer_id', how='left')`


In [2]:
cust_meta = pd.DataFrame({
    'customer_id': df['customer_id'].unique()[:100],
    'credit_tier': np.random.choice(['Tier1', 'Tier2', 'Tier3'], size=100),
    'loyalty_years': np.random.randint(1, 10, size=100)
})
enriched_tx = pd.merge(df.head(20), cust_meta, on='customer_id', how='left')
print('Enriched Transactions Head:\n', enriched_tx[['transaction_id', 'customer_id', 'transaction_amount', 'credit_tier']].head())

Enriched Transactions Head:
   transaction_id customer_id  transaction_amount credit_tier
0       TX110686      C82845             1216.33       Tier2
1       TX107170      C85674              324.99       Tier1
2       TX108328      C32431              136.66       Tier2
3       TX108563      C54057              124.21       Tier3
4       TX107002      C95649             1284.68       Tier3


### 🔹 Vertical Batch Stacking with `pd.concat(axis=0)`
Combines split transaction batches row-wise. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Always check `df.shape` before and after merging to verify you didn't accidentally introduce Cartesian duplication or drop rows.

**Syntax:** `pd.concat([batch1, batch2], axis=0, ignore_index=True)`


In [3]:
batch_a = df.iloc[0:50]
batch_b = df.iloc[50:100]
stacked_batches = pd.concat([batch_a, batch_b], axis=0, ignore_index=True)
print('Vertically Stacked Rows Total:', len(stacked_batches))

Vertically Stacked Rows Total: 100


### 🔹 Horizontal Feature Concatenation with `pd.concat(axis=1)`
Merges transaction features horizontally along row indices. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Always check `df.shape` before and after merging to verify you didn't accidentally introduce Cartesian duplication or drop rows.

**Syntax:** `pd.concat([df_numeric, df_categorical], axis=1)`


In [4]:
num_feats = df[['transaction_amount', 'account_age_months']].head(5)
cat_feats = df[['card_type', 'device_type']].head(5)
horiz_df = pd.concat([num_feats, cat_feats], axis=1)
print('Horizontally Combined Features:\n', horiz_df)

Horizontally Combined Features:
    transaction_amount  account_age_months   card_type device_type
0             1216.33                  56        Visa         POS
1              324.99                 112  MasterCard     Desktop
2              136.66                  68    Discover         ATM
3              124.21                  50        Amex         POS
4             1284.68                  96  MasterCard         ATM


### 🔹 Index-Based Joins with `df.join()`
Joins auxiliary merchant statistics indexed by `merchant_id`. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Always check `df.shape` before and after merging to verify you didn't accidentally introduce Cartesian duplication or drop rows.

**Syntax:** `df.set_index('merchant_id').join(merchant_risk_df)`


In [5]:
merchant_stats = df.groupby('merchant_id')['is_fraud'].mean().rename('merchant_fraud_rate')
joined_tx = df.head(10).set_index('merchant_id').join(merchant_stats)
print('Merchant-Joined Transactions Head:\n', joined_tx[['transaction_id', 'merchant_fraud_rate']].head())

Merchant-Joined Transactions Head:
             transaction_id  merchant_fraud_rate
merchant_id                                    
M2697             TX110686             0.032258
M3868             TX107170             0.075000
M1461             TX108328             0.032258
M1713             TX108563             0.148148
M1474             TX107002             0.137931


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Merge Validation & Cardinality Integrity

**Approach:** Validate that a customer metadata merge is strictly many-to-one using `validate='m:1'` and check for dropped records with `indicator=True`.
**Syntax:** `pd.merge(df, cust_meta, on='customer_id', how='left', validate='m:1', indicator=True)`


In [6]:
checked_merge = pd.merge(df.head(50), cust_meta, on='customer_id', how='left', validate='m:1', indicator=True)
print('Merge Match Status Breakdown:\n', checked_merge['_merge'].value_counts())

Merge Match Status Breakdown:
 _merge
both          50
left_only      0
right_only     0
Name: count, dtype: int64
